In [ ]:
%%capture
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dj_notebook import activate
from django_pandas.io import read_frame

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
from datetime import datetime

from edc_lab.dataframes import get_requisition_df
from edc_lab_results_import.result_importer import ResultImporter
from edc_registration.models import RegisteredSubject

from meta_subject.models import BloodResultsIns
from meta_subject.models import Glucose, GlucoseFbg
from edc_pdutils.dataframes import get_subject_visit
from meta_screening.models import SubjectScreening
from meta_rando.models import RandomizationList
from meta_prn.models import EndOfStudy

In [ ]:
export_to_file = True
visit_code = 1000.0

In [ ]:
def get_screening_df()-> pd.DataFrame:
    values = [
        "screening_identifier",
        "subject_identifier",
        "report_datetime",
        "fbg_value",
        "fbg_units",
        "fbg_datetime",
        "converted_fbg_value",
        "fbg2_value",
        "fbg2_units",
        "fbg2_datetime",
        "converted_fbg2_value",
        'ogtt_units',
        'ogtt_base_datetime',
        'ogtt_value',
        'ogtt_quantifier',
        'ogtt_datetime',
        'ogtt2_units',
        'ogtt2_base_datetime',
        'ogtt2_value',
        'ogtt2_quantifier',
        'ogtt2_datetime',
        "fasting",
        "repeat_fasting",
        "fasting_duration_delta",
        "repeat_fasting_duration_delta",
        'fasting',
        'fasting_duration_str',
        'fasting_duration_delta',
        'repeat_fasting',
        'repeat_fasting_duration_str',
        'repeat_fasting_duration_delta',
        'site',
    ]
    df:pd.DataFrame = (
        read_frame(SubjectScreening.objects.values(*values).all(), verbose=False)
        .rename(columns={
            "report_datetime":"screening_datetime",
            "fbg_value": "fbg1_value",
            "fbg_units": "fbg1_units",
            "converted_fbg_value":"converted_fbg1_value",
            "fasting": "fasting1",
            "repeat_fasting": "fasting2",
            "fasting_duration_delta": "fasting1_duration_delta",
            "fasting_duration_str": "fasting1_duration_str",
            "repeat_fasting_duration_delta": "fasting2_duration_delta",
            "repeat_fasting_duration_str": "fasting2_duration_str",
            "fbg_datetime": "fbg1_datetime",
            "ogtt_value": "ogtt1_value",
            "ogtt_units": "ogtt1_units",
            "ogtt_quantifier": "ogtt1_quantifier",
            "ogtt_base_datetime": "ogtt1_base_datetime",
            "ogtt_datetime": "ogtt1_datetime",
        })
    )
    for col in ["fbg1_value", "converted_fbg1_value", "fbg2_value", "converted_fbg2_value", 'ogtt1_value','ogtt2_value',]:
        df[col] = df[col].astype("Float64").fillna(pd.NA)

    cols = [
        "screening_identifier",
        "subject_identifier",
        "fbg1_units",
        "fbg2_units",
        'ogtt1_units',
        'ogtt1_quantifier',
        'ogtt2_units',
        'ogtt2_quantifier',
        "fasting1",
        "fasting2",
        'fasting1_duration_str',
        'fasting2_duration_str',
    ]
    for col in cols:
        df[col] = df[col].astype("string").fillna(pd.NA)

    for col in [c for c in df.select_dtypes(include="datetimetz")]:
        df[col] = df[col].dt.tz_convert("utc").dt.normalize().dt.tz_localize(None)

    df["fbg_value"] = np.where(
        df["fbg2_value"].notna(),
        df["fbg2_value"],
        df["fbg1_value"]
    )

    df["converted_fbg_value"] = np.where(
        df["converted_fbg2_value"].notna(),
        df["converted_fbg2_value"],
        df["converted_fbg1_value"]
    )

    df["fasting"] = np.where(
        df["fbg2_value"].notna(),
        df["fasting2"],
        df["fasting1"]
    )

    df["fbg_units"] = np.where(
        df["fbg2_units"].notna(),
        df["fbg2_units"],
        df["fbg1_units"]
    )

    df["fasting_duration_delta"] = np.where(
        df["fbg2_value"].notna(),
        df["fasting2_duration_delta"],
        df["fasting1_duration_delta"]
    )

    df["fbg_datetime"] = np.where(
        df["fbg2_datetime"].notna(),
        df["fbg2_datetime"],
        df["fbg1_datetime"]
    )

    df["fasting_hrs"] = df["fasting_duration_delta"].apply(lambda x: x.total_seconds() / 3600)
    df = df.loc[df["subject_identifier"].str.startswith("105-")]
    return df.loc[(df["fasting_duration_delta"] >= pd.Timedelta(hours=8.0))].reset_index(drop=True)


In [ ]:
df_screening = get_screening_df()

In [ ]:
df_screening.screening_datetime.agg(["min", "max"])

In [ ]:
df_registered_subject = read_frame(RegisteredSubject.objects.values("subject_identifier", "gender", "dob", "randomization_datetime").all(), verbose=False)
# df_registered_subject.randomization_datetime.agg(["min", "max"])
subject_identifiers = df_registered_subject.subject_identifier

In [ ]:
df_visit = get_subject_visit(model="meta_subject.subjectvisit").rename(columns={"subject_visit_id": "subject_visit", "report_datetime": "visit_datetime"}).query("reason=='scheduled'")[["subject_identifier", "subject_visit", "visit_datetime", "site_id", "baseline_datetime", "visit_count_total"]].reset_index(drop=True)

In [ ]:
df_requisitions = get_requisition_df()
df_requisitions["drawn_datetime"] = df_requisitions["drawn_datetime"].dt.tz_convert("utc").dt.normalize()

In [ ]:
df = df_visit.merge(df_requisitions[["subject_visit", "visit_code", "requisition", "requisition_identifier", "panel_name", "utestid", "drawn_datetime"]], on=["subject_visit"], how="left", suffixes=["", "_y"])
df = df.drop(columns=[c for c in df.columns if c.endswith("_y")]).reset_index(drop=True).query(f"utestid=='ins' and visit_code<={visit_code + 10.0}")
# df

In [ ]:
df = df.merge(df_screening[["subject_identifier", "screening_identifier", "screening_datetime", "fbg_value", "fbg_units", "fbg_datetime", "fasting_hrs"]], on="subject_identifier", how="left")

In [ ]:
df_ins = read_frame(BloodResultsIns.objects.all(), verbose=False).rename(columns={"ins_value": "insulin_keyed"})
df_ins["requisition"] = df_ins["requisition"].astype("string").fillna(pd.NA)
df_ins["assay_datetime"] = df_ins["assay_datetime"].dt.tz_convert("utc").dt.normalize()
df = df.merge(df_ins[["requisition", "insulin_keyed"]], on="requisition", how="left", suffixes=["", "_y"])
# df = df[(df["visit_code"]==1000.0) & (df["utestid"] == "ins")][["subject_identifier", "subject_visit", "result_value", "visit_code", "visit_code_sequence", "visit_datetime", "drawn_datetime", "panel_name", "utestid"]].copy().reset_index(drop=True)

In [ ]:
df

In [ ]:
df_results_orig = ResultImporter.model_to_dataframe()
df_results = df_results_orig.copy().rename(columns={"result_value": "insulin_imported", "specimen_collected_datetime": "insulin_imported_datetime"}).query("utestid=='ins'").reset_index(drop=True)

In [ ]:
#df_results

In [ ]:
df0 = df.copy().reset_index(drop=True)
df1 = df0.merge(df_results[["requisition", "insulin_imported", "insulin_imported_datetime"]], on="requisition", how="left", suffixes=["", "_y"])
df1 = df1[~df1["insulin_imported"].isna()].reset_index(drop=True)
df2 = df0.merge(df_results[["screening_identifier", "insulin_imported", "insulin_imported_datetime"]], on="screening_identifier", how="left", suffixes=["", "_y"])
df2 = df2[~df2["subject_identifier"].isin(df1.subject_identifier)]

df3 = pd.concat([df1,df2], ignore_index=True).reset_index(drop=True)

df0 = df0.merge(df3[["subject_identifier", "insulin_imported", "insulin_imported_datetime"]], on="subject_identifier", how="left").sort_values(by=["subject_identifier", ])
df0["insulin_keyed"] = df0["insulin_keyed"].astype("Float64").fillna(pd.NA)
df0["insulin_imported"] = df0["insulin_imported"].astype("Float64").fillna(pd.NA).round(2)

df0["insulin"] = df0["insulin_keyed"]
df0.loc[(df0["insulin"].isna() & ~df0["insulin_imported"].isna()), "insulin"] = df0["insulin_imported"]
df0.loc[(df0["insulin_keyed"] != df0["insulin_imported"]), "insulin"] = df0["insulin_imported"]

df0.sort_values(by="subject_identifier").drop_duplicates(subset=["subject_identifier"], keep='first')

df_export = df0[["subject_identifier", "fbg_value", "insulin"]].rename(columns={"fbg_value": "glucose"})

tstamp = datetime.today().strftime('%Y%m%d%H%M')
df_export.to_parquet(analysis_folder / f"homa2_data_{tstamp}.parquet", index=False)

In [ ]:
df_export